# Cyanite model outputs (tagging)

Fetch Cyanite's per-track audio analysis (the tags) from the **model outputs** endpoint.
This is the core of the *"why this track?"* explainability: every indexed track has rich,
human-readable outputs (genre, mood, instruments, character, BPM, energy, era, and more).

- Endpoint: `GET {BASE_URL}/library-tracks/{track_id}/models?model=...`
- Model reference: [../guides/model_outputs.md](../guides/model_outputs.md); tag values: [../guides/tag_vocabularies.md](../guides/tag_vocabularies.md)

Search (free-text and similarity) is scaffolded at the bottom to fill in later.

## Setup

```bash
pip install requests python-dotenv
```

Put your credentials in a `.env` file (git-ignored, never commit it):

```
CYANITE_API_KEY=your_key_here
CYANITE_ACCOUNT=your_account_id   # used by the search endpoints
```

In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads CYANITE_API_KEY (and CYANITE_ACCOUNT) from .env

API_KEY = os.environ["CYANITE_API_KEY"]
ACCOUNT = os.environ.get("CYANITE_ACCOUNT")  # needed for search; not for model outputs
BASE_URL = "https://rest-api.cyanite.ai/v1"

session = requests.Session()
session.headers.update({"x-api-key": API_KEY})
print("ready")

## Resolve track ids: Jamendo -> Cyanite

The data pack (`data/users.csv`, `data/tracks.csv`) uses **Jamendo** numeric ids, but the
API is keyed on **Cyanite** track ids (`libtr_...`). `data/jamendo_mapper.json` provides the
mapping. It is keyed by Cyanite id, e.g.

```json
{"libtr_01KVWSKNRN6XJMTVV0": {"jamendo_title": "1000002.mp3", "status": "AVAILABLE"}}
```

so we invert it once to look up a Cyanite id from a Jamendo id. (A few catalog tracks may be
missing or not `AVAILABLE`.)

In [ ]:
def _find_mapper():
    for p in (Path("../data/jamendo_mapper.json"), Path("data/jamendo_mapper.json")):
        if p.exists():
            return p
    raise FileNotFoundError("jamendo_mapper.json not found under ../data or data")

with open(_find_mapper()) as f:
    _mapper = json.load(f)

# Jamendo track id (string) -> Cyanite track id, for AVAILABLE tracks
_jamendo_to_cyanite = {
    info["jamendo_title"].rsplit(".", 1)[0]: cyanite_id
    for cyanite_id, info in _mapper.items()
    if info.get("status") == "AVAILABLE"
}
print(f"loaded {len(_jamendo_to_cyanite):,} Jamendo -> Cyanite mappings")


def resolve_track_id(jamendo_id):
    """Map a Jamendo track id (from the data pack) to a Cyanite track id, or None."""
    return _jamendo_to_cyanite.get(str(jamendo_id))

## The model outputs endpoint

Pass one or more `model` query parameters to select which outputs to return. Full list in the
[model outputs guide](../guides/model_outputs.md); the ones below are the most useful here.

In [ ]:
DEFAULT_MODELS = [
    "MainGenreV2", "SubgenreV2", "FreeGenreV3",
    "MoodSimpleV2", "MoodAdvancedV2",
    "InstrumentsV2", "CharacterV2", "MovementV2",
    "BpmV2", "TempoV1", "KeyV2", "TimeSignatureV2",
    "ValenceArousalV2", "MusicalEraV2", "MusicForV1",
    "VocalsV2", "AutoDescriptionV2", "RepresentativeSegmentV2",
]


def get_model_outputs(track_id, models=None):
    """Fetch selected model outputs for a Cyanite track id (libtr_...)."""
    models = models or DEFAULT_MODELS
    params = [("model", m) for m in models]
    resp = session.get(f"{BASE_URL}/library-tracks/{track_id}/models", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

In [ ]:
# End to end: start from a Jamendo id (as found in the data pack), resolve, fetch tags.
jamendo_id = "1000002"
track_id = resolve_track_id(jamendo_id)
print(jamendo_id, "->", track_id)

outputs = get_model_outputs(track_id, ["MainGenreV2", "MoodSimpleV2", "InstrumentsV2", "BpmV2", "AutoDescriptionV2"])
print(json.dumps(outputs, indent=2)[:2500])

## Summarize the tags

A small helper to pull the human-readable bits out of the response for explanations. Adjust
the accessors once you see a real response (the model outputs guide documents each field).

In [ ]:
def iter_model_outputs(outputs):
    """Yield individual model-output dicts regardless of envelope shape."""
    if isinstance(outputs, dict):
        for key in ("data", "models", "results"):
            if isinstance(outputs.get(key), list):
                yield from outputs[key]
                return
        for v in outputs.values():
            if isinstance(v, dict) and "version" in v:
                yield v
        return
    if isinstance(outputs, list):
        yield from outputs


def summarize(outputs):
    """Compact human-readable summary: dominant tags / values per model."""
    summary = {}
    for mo in iter_model_outputs(outputs):
        version = mo.get("version", "?")
        if mo.get("tags") is not None:
            summary[version] = mo["tags"]
        elif "tag" in mo:
            summary[version] = mo["tag"]
        elif "description" in mo:
            summary[version] = mo["description"]
    return summary


summarize(outputs)

## Free-text search  (to add)

Natural language -> ranked track ids (+ scores), optionally with tag filters. Endpoint TBD
(Search 2.0); it takes an `accountID` (use `ACCOUNT`). Pattern: search to get track ids,
then `get_model_outputs` on the ones you show, and explain via their tags.

In [ ]:
def free_text_search(query, limit=20, filters=None, account=ACCOUNT):
    """Free-text / prompt search -> ranked track ids with scores.

    TODO: wire up the Search 2.0 free-text endpoint (path, request body, response shape).
    Expected to return something like: [{"track_id": ..., "score": ...}, ...].
    """
    raise NotImplementedError("Free-text search endpoint TBD.")

## Similarity search  (to add)

Given a seed track id, return acoustically similar track ids (+ scores). Combine with
`get_model_outputs` to re-rank for tag coherence and to explain each pick.

In [ ]:
def similarity_search(track_id, limit=20, filters=None, account=ACCOUNT):
    """Similar-by-id -> ranked track ids with scores.

    TODO: wire up the Search 2.0 similarity endpoint (path, request body, response shape).
    """
    raise NotImplementedError("Similarity search endpoint TBD.")